## CS3570 Introduction to Multimedia Technology
### Homework 03 Due: 11:59pm, 2026/05/11

### Reminder:
* **You are not allowed to use any function that directly generates the result for each step, except for those provided by us.**
* **Your code must display and output your results to enable us to verify its correctness.**
* **Please follow the instructions in the Jupyter Notebook and complete the parts marked as `"TODO."`**
* **Please include your report in the Jupyter Notebook using markdown.**
* **Please compress your code, input images, and result images in a zip file named HW3_{StudentID}.zip and upload it to eeclass.**
* **Homework should be submitted before the announced due time. Scores of late homework will be reduced by 20% per day.**
* **If you encounter any problems or have questions, please post them on eeclass.**

## Part 1: Bézier Curve (50%)

In [ ]:
import numpy as np
import cv2
import os
import matplotlib.pyplot as plt

In [ ]:
if not os.path.exists('output'):
    os.makedirs('output')

In [ ]:
# Utility functions
def draw_subplot(points, result1, result2, img):
    dpi = 200
    h, w = img.shape[:2]
    plt.figure(figsize=(w / dpi, h / dpi))
    plt.imshow(img)
    plt.scatter(points[:, 0], points[:, 1], s=1, color='blue')
    plt.plot(result1[:, 0], result1[:, 1], linewidth=0.7, color='cyan')
    plt.plot(result2[:, 0], result2[:, 1], linewidth=0.7, color='magenta')
    plt.savefig('output/1a.png', dpi=dpi, bbox_inches='tight')
    plt.show()


def draw_plot(points, result, img):
    dpi = 200
    h, w = img.shape[:2]
    plt.figure(figsize=(w / dpi, h / dpi))
    plt.imshow(img)
    plt.scatter(points[:, 0], points[:, 1], s=10, color='blue')
    plt.plot(result[:, 0], result[:, 1], linewidth=3, color='magenta')
    plt.savefig('output/1b.png', dpi=dpi, bbox_inches='tight')
    plt.show()

In [ ]:
# TODO: Implement the cubic Bézier curve calculation.
#       You may adjust the function if needed.
def bezier_curve(points, step):
    curve_points = []

    # t values from 0 to 1
    t_values = np.arange(0, 1 + step, step)

    # next curve starts every 3 points
    for i in range(0, len(points) - 1, 3):
        p0, p1, p2, p3 = points[i:i + 4]

        for t in t_values:
            # cubic Bézier formula:
            # B(t) = (1-t)^3 p0 + 3t(1-t)^2 p1 + 3t^2(1-t) p2 + t^3 p3
            point = ((1 - t) ** 3) * p0 + 3 * t * ((1 - t) ** 2)  * p1 + 3 * (t ** 2) * (1 - t) * p2 + (t ** 3) * p3

            curve_points.append(point)

    return np.array(curve_points)

# TODO: Implement nearest-neighbor interpolation to scale up the image by 4x.
#       You may adjust the function if needed.
def nearest_inter(img, scale = 4):
    scale = int(scale)

    # original image h and w
    height, width = img.shape[:2]

    # size of scaled image
    new_height = height * scale
    new_width = width * scale

    # for each pixel position in scaled image, find row and col index in orig. image
    old_y = np.arange(new_height) // scale
    old_x = np.arange(new_width) // scale

    # copy pixels from orig. image to scaled image.
    scaled = img[old_y[:, None], old_x[None, :]]

    return scaled


### 1(a) Implement the Cubic Bézier Curves (30%)
Plot two sets of cubic Bézier curves with different detail levels, along with their control points, onto `bg.png`. Each set contains $(N-1)/3$ cubic Bézier curves connected end-to-end.
- Low detail curve: $t = \{0, 0.25, 0.5, 0.75, 1.0\}$
- High detail curve: $t = \{0, 0.001, 0.002, \ldots, 1.0\}$

In [ ]:
img = cv2.imread("./bg.png")
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
points = np.loadtxt("./points.txt")

# TODO: Compute the low-detail and high-detail Bézier curves using bezier_curve()
result1 = bezier_curve(points, 0.25)  # low-detail curves
result2 = bezier_curve(points, 0.001)  # high-detail curves

draw_subplot(points, result1, result2, img)

### 1(b) Scale Up the Image and Curves by 4× (10%)

In [ ]:
# TODO: Scale up the image and the Bézier curve

scaled_img = nearest_inter(img, 4) # 4x scaled image via nearest-neighbor interpolation
scaled_points = points * 4 # 4x scaled control points
scaled_result = bezier_curve(scaled_points, 0.001) # high-detail Bézier curve on the scaled points

draw_plot(scaled_points, scaled_result, scaled_img)

### 1(c) Report (10%)
# I. Bézier curve calculation

For the Bézier curve part, I calculated the curve using the cubic Bézier formula. Each curve segment uses 4 control points. The first and last points are the starting and ending points of the curve, while the two middle points control how the curve bends.

In the code, for every value of t (from 0 to 1 according to step), I calculated one point on the curve. After getting all the points, I plotted them together so they form the final Bézier curve.

For the low-detail version,a bigger step size is used so there were fewer points on the curve. For the high-detail version, a smaller step size is used so more points were calculated and the curve became smoother.


# II. High-detail and low-detail curves

The main difference I noticed is the smoothness. The low-detail curve still follows the same general shape, but it looks more rough because there are not many points. Some parts look like straight line segments instead of a smooth curve.

The high-detail curve looks much smoother because it has a lot more points. The shape is also closer to the actual Bézier curve. However, it also needs more calculations compared to the low-detail one.

So basically, the **low-detail curve is simpler and faster, but the quality is lower**. The **high-detail curve looks better visually, but it uses more points and takes more computation.**


# III. Upscaling raster image and vector curves

For the background image, I upscaled it using nearest-neighbor interpolation. This makes the image bigger by copying the nearest pixel value. Because of that, the image becomes blocky and pixelated after scaling. It does not really create new details, it only enlarges the old pixels.

For the vector curve, the result is different. The curve is based on mathematical control points, so when I scale it, I can scale the points and draw the curve again. Because of this, the curve can stay smooth even after being enlarged.

From this part, I can see that raster images and vector graphics behave differently when scaled. **Raster images depend on pixels, so they can lose quality when enlarged. Vector curves depend on equations and points, so they are better for scaling.**

## Part 2: 3D Models (40%)

In [ ]:
# Packages come from the repo-root requirements.txt.
# Uncomment the next line only if you are running this notebook by itself:
# %pip install meshio plotly "kaleido==0.2.1" ipykernel nbformat

In [ ]:
# Import 3D libraries
import meshio
import plotly.offline as py
import plotly.graph_objs as go
import plotly.io as pio
import plotly.subplots as sp

# Set notebook mode to work in offline
#py.init_notebook_mode()
#pio.renderers.default = 'iframe'
#Try uncommenting the above 2 lines if you can't correctly show the 3D rendering

### Load the bunny model

In [ ]:
msh = meshio.read("bunny.obj")
verts = msh.points
x, y, z = verts.T
I, J, K = msh.cells_dict["triangle"].T
colorscale = [[0, 'rgb(255,192,203)'],
              [1, 'rgb(255,192,203)']]

In [ ]:
trace = go.Mesh3d(x=x, y=y, z=z,
                i=I, j=J, k=K,
                intensity=z,
                colorscale=colorscale,
                showscale=False)

fig = go.Figure(trace)
fig.show()

### 2(a) Model Transformation (30%)
Position and orient the bunny such that it satisfies **both** of the following conditions:
1. The center of the bunny is placed at the origin $(0,0,0)$.
2. The bunny is facing directly toward the screen.

In [ ]:
# TODO: Implement the geometric transformations to center and rotate the bunny.
# put all vertices together
vertices = np.column_stack((x, y, z))

# center is defined as [(max(x) + min(x)) / 2, (max(y) + min(y)) / 2, (max(z) + min(z)) / 2]
center = (vertices.max(axis=0) + vertices.min(axis=0)) / 2.0

# center bunny to origin
v = vertices - center

# rotate +90 degrees around x-axis
# original: x, y, z
# after +90° x-rotation: x' = x, y' = -z, z' = y
x_transform = -v[:, 0]
y_transform = v[:, 2]
z_transform = v[:, 1]

trace = go.Mesh3d(x=x_transform, y=y_transform, z=z_transform,
                i=I, j=J, k=K,
                intensity=z,
                colorscale=colorscale,
                showscale=False)

fig = go.Figure(trace)
fig.write_image("output/2a.png", scale=2)
fig.show()

### 2(b) Surface Lighting (5%)
Render the bunny under different illumination settings by varying the ambient strength $k_a$, diffuse strength $k_d$, and specular strength $k_s$:

In [ ]:
# Example of applying lighting to the transformed bunny
fig = go.Figure(go.Mesh3d(
    x=x_transform, y=y_transform, z=z_transform,
    i=I, j=J, k=K,
    intensity=z_transform,
    colorscale=colorscale,
    lighting=dict(ambient=0.6, diffuse=0.5, specular=0.3),
    showscale=False
))
fig.show()

In [ ]:
# TODO: Compare different ambient, diffuse and specular lighting settings on the transformed bunny from 2(b).

lighting_settings = [
    [dict(ambient=0.1, diffuse=0.5, specular=0.05),
     dict(ambient=0.8, diffuse=0.5, specular=0.05)],

    [dict(ambient=0.4, diffuse=0.1, specular=0.05),
     dict(ambient=0.4, diffuse=0.9, specular=0.05)],

    [dict(ambient=0.5, diffuse=0.8, specular=0.1),
     dict(ambient=0.5, diffuse=0.8, specular=0.8)],
]
for i, light in enumerate(lighting_settings):

    fig = sp.make_subplots(rows=1, cols=2, specs=[[{'type': 'surface'}, {'type': 'surface'}]])

    trace1 = go.Mesh3d(x=x_transform, y=y_transform, z=z_transform,
                i=I, j=J, k=K,
                intensity=z_transform,
                colorscale =colorscale,
                showscale=False,
                lighting=dict(ambient=light[0]["ambient"], diffuse=light[0]["diffuse"], specular=light[0]["specular"]))
    fig.add_trace(trace1, row=1, col=1)

    trace2 = go.Mesh3d(x=x_transform, y=y_transform, z=z_transform,
                i=I, j=J, k=K,
                intensity=z_transform,
                colorscale =colorscale,
                showscale=False,
                lighting=dict(ambient=light[1]["ambient"], diffuse=light[1]["diffuse"], specular=light[1]["specular"]))
    fig.add_trace(trace2, row=1, col=2)

    fig.write_image(f"output/2b_{i+1}.png")
    fig.show()

### 2(c) Report (15%)
# I. Transformations in 2(a)
In 2(a), I transformed the bunny model so it is centered and facing the screen better:
1. calculate the center of the bunny using the maximum and minimum values of the coordinates (formula is commented and based from the PDF).
2. subtracted the center from all vertices, so the bunny moved to around the origin.
3. After that, I changed the coordinate directions to adjust how the bunny is facing. In my code, I used the transformed coordinates like this:


```
x_transform = -v[:, 0]y_transform = v[:, 2]z_transform = v[:, 1]
```

This changes the axes and makes the bunny face the screen in a better direction.

IMPORTANT NOTE: The order of transformations does matter. For example, translating first and rotating later can give a different result compared to rotating first and translating later. This is because rotation depends on the position of the object. If the object is not centered first, it may rotate around a different point and the final position may look wrong. In my implementation, I centered the bunny first, then adjusted the orientation, because this makes the transformation easier to control.

# II. Lighting settings in 2(b)
For the lighting part, I compared three pairs of lighting settings. Each pair changes one main lighting component, so it is easier to see the effect.
1. For the first pair, the ambient value is changed from low to high.


```
[dict(ambient=0.1, diffuse=0.5, specular=0.05),
 dict(ambient=0.8, diffuse=0.5, specular=0.05)],
```


When the ambient value is low, the bunny looks darker overall. Some parts are harder to see. When the ambient value is high, the whole bunny becomes brighter, including the parts that are not directly facing the light. So I think ambient lighting mainly controls the general brightness of the object.

2. For the second pair, the diffuse value is changed.
```
[dict(ambient=0.4, diffuse=0.1, specular=0.05),
 dict(ambient=0.4, diffuse=0.9, specular=0.05)],
```

When the diffuse value is low, the bunny looks flatter and the shape is not very clear. When the diffuse value is high, the surfaces facing the light become brighter, so the 3D shape is easier to notice. This makes the bunny look more solid. Diffuse lighting is important because it shows how the surface direction reacts to the light.

3. For the third pair, the specular value is changed.
```
[dict(ambient=0.5, diffuse=0.8, specular=0.1),
 dict(ambient=0.5, diffuse=0.8, specular=0.8)],
```
When the specular value is low, the bunny looks more matte and not shiny. When the specular value is high, there are stronger bright highlights on the surface. This makes the bunny look more glossy or reflective. So specular lighting mainly affects the shiny parts of the object.

**Overall, ambient lighting changes the overall brightness, diffuse lighting helps show the 3D shape, and specular lighting controls the shiny highlights. By comparing the pairs, I can see that changing each lighting component gives a different visual effect on the bunny.**